In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bestrobertaforgivingaspect/config.json
/kaggle/input/bestrobertaforgivingaspect/merges.txt
/kaggle/input/bestrobertaforgivingaspect/training_args.bin
/kaggle/input/bestrobertaforgivingaspect/vocab.json
/kaggle/input/bestrobertaforgivingaspect/tokenizer_config.json
/kaggle/input/bestrobertaforgivingaspect/model.safetensors
/kaggle/input/bestrobertaforgivingaspect/special_tokens_map.json
/kaggle/input/aspectdatasetforgivingaspect/Udemy_grouped_by_categoryforaspect.csv
/kaggle/input/roberta-finutuneonpseudoandaugmentation/config.json
/kaggle/input/roberta-finutuneonpseudoandaugmentation/merges.txt
/kaggle/input/roberta-finutuneonpseudoandaugmentation/training_args.bin
/kaggle/input/roberta-finutuneonpseudoandaugmentation/vocab.json
/kaggle/input/roberta-finutuneonpseudoandaugmentation/tokenizer_config.json
/kaggle/input/roberta-finutuneonpseudoandaugmentation/model.safetensors
/kaggle/input/roberta-finutuneonpseudoandaugmentation/special_tokens_map.json


In [2]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import nltk
from nltk.tokenize import sent_tokenize
from collections import Counter
from tqdm import tqdm

nltk.download('punkt', quiet=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading dataset (Chunk 1)...")
df = pd.read_csv('/kaggle/input/aspectdatasetforgivingaspect/Udemy_grouped_by_categoryforaspect.csv', nrows=500000)

print("Loading model and tokenizer...")
model_path = "/kaggle/input/bestrobertaforgivingaspect"
tokenizer = RobertaTokenizer.from_pretrained(model_path)
model = RobertaForSequenceClassification.from_pretrained(model_path)

model.to(device)

label_to_aspect = {
    0: 'Ambiguous', 1: 'Assignment & Tutorial', 2: 'Design & Material',
    3: 'Financial', 4: 'Instructor', 5: 'Platform & Technical', 6: 'Skill & Career'
}

def predict_aspect(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    return label_to_aspect[predicted_class]

print("Processing comments and predicting aspects (Chunk 1)...")
new_rows = []
all_aspects = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing comments"):
    sentences = sent_tokenize(row['comment'])
    for sentence in sentences:
        aspect = predict_aspect(sentence)
        all_aspects.append(aspect)
        new_row = row.copy()
        new_row['sentence'] = sentence
        new_row['predicted_aspect'] = aspect
        new_rows.append(new_row)

new_df = pd.DataFrame(new_rows)

new_df.to_csv('udemy_comments_with_aspects_chunk1.csv', index=False)
print("Chunk 1 processed and saved to 'udemy_comments_with_aspects_chunk1.csv'")

aspect_counts = Counter(all_aspects)
total_sentences = len(all_aspects)
results_df = pd.DataFrame([(aspect, count, (count/total_sentences)*100) 
                           for aspect, count in aspect_counts.most_common()],
                          columns=['Aspect', 'Count', 'Percentage'])
results_df.to_csv('aspect_distribution_results_chunk1.csv', index=False)
print("Aspect distribution for Chunk 1 saved to 'aspect_distribution_results_chunk1.csv'")

Using device: cuda:0
Loading dataset (Chunk 1)...
Loading model and tokenizer...
Processing comments and predicting aspects (Chunk 1)...


Processing comments: 100%|██████████| 500000/500000 [3:29:34<00:00, 39.76it/s]


Chunk 1 processed and saved to 'udemy_comments_with_aspects_chunk1.csv'
Aspect distribution for Chunk 1 saved to 'aspect_distribution_results_chunk1.csv'
